In [2]:
import cv2
import numpy as np
import pandas as pd
from scipy.fftpack import dct
import os

def extract_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (128, 128))
    dct_coeffs = dct(dct(img.T, norm='ortho').T, norm='ortho')
    return {
        'mean': np.mean(dct_coeffs),
        'std': np.std(dct_coeffs),
        'variance': np.var(dct_coeffs),
        'skewness': float(np.mean((dct_coeffs - np.mean(dct_coeffs))**3)),
        'energy': np.sum(dct_coeffs**2)
    }

# Load images from both folders
rows = []
for label, folder in [
    (0, '../data/images/train/train/clean'),
    (1, '../data/images/train/train/stego')
]:
    files = os.listdir(folder)[:200]  # 200 from each = 400 total
    for f in files:
        path = os.path.join(folder, f)
        feats = extract_features(path)
        if feats:
            feats['label'] = label  # 0=clean, 1=stego
            rows.append(feats)

df = pd.DataFrame(rows)
df.to_csv('../models/features_dataset.csv', index=False)
print(f"Dataset built: {len(df)} images")
print(df['label'].value_counts())
print(df.head())

Dataset built: 400 images
label
0    200
1    200
Name: count, dtype: int64
       mean         std      variance      skewness       energy  label
0  0.068344  160.416491  25733.450468  2.231184e+08  421616929.0      0
1  0.048488  110.018753  12104.126006  8.149722e+07  198314039.0      0
2  0.077375  164.950014  27208.507136  3.051740e+08  445784279.0      0
3  0.076428  171.317650  29349.737201  3.325636e+08  480866190.0      0
4  0.028788   64.258754   4129.187526  1.582861e+07   67652622.0      0


In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Load the dataset you just built
df = pd.read_csv('../models/features_dataset.csv')

X = df.drop('label', axis=1)
y = df['label']

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['clean', 'stego']))

# Save the model
joblib.dump(clf, '../models/image_classifier.pkl')
print("\nModel saved!")

c:\Users\lizag\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\lizag\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Accuracy: 0.5625

Classification Report:
              precision    recall  f1-score   support

       clean       0.63      0.50      0.56        44
       stego       0.51      0.64      0.57        36

    accuracy                           0.56        80
   macro avg       0.57      0.57      0.56        80
weighted avg       0.58      0.56      0.56        80


Model saved!


In [3]:
import cv2
import numpy as np
import pandas as pd
import os
import joblib
from scipy.fftpack import dct
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
# Add more features to improve accuracy
def extract_features_v2(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (128, 128))
    dct_coeffs = dct(dct(img.T, norm='ortho').T, norm='ortho')
    
    # Original features
    feats = {
        'mean': np.mean(dct_coeffs),
        'std': np.std(dct_coeffs),
        'variance': np.var(dct_coeffs),
        'skewness': float(np.mean((dct_coeffs - np.mean(dct_coeffs))**3)),
        'energy': np.sum(dct_coeffs**2),
        # New features
        'entropy': -np.sum((np.abs(dct_coeffs)/np.sum(np.abs(dct_coeffs)+1e-10)) * 
                   np.log2(np.abs(dct_coeffs)/np.sum(np.abs(dct_coeffs)+1e-10) + 1e-10)),
        'kurtosis': float(np.mean((dct_coeffs - np.mean(dct_coeffs))**4)),
        'max_coeff': np.max(np.abs(dct_coeffs)),
        'pixel_mean': np.mean(img),
        'pixel_std': np.std(img)
    }
    return feats

# This time use 500 images from each class
rows = []
for label, folder in [
    (0, '../data/images/train/train/clean'),
    (1, '../data/images/train/train/stego')
]:
    files = os.listdir(folder)[:500]
    for f in files:
        path = os.path.join(folder, f)
        feats = extract_features_v2(path)
        if feats:
            feats['label'] = label
            rows.append(feats)

df2 = pd.DataFrame(rows)

X = df2.drop('label', axis=1)
y = df2['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf2 = RandomForestClassifier(n_estimators=200, random_state=42)
clf2.fit(X_train, y_train)

y_pred = clf2.predict(X_test)
print("Improved Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['clean', 'stego']))

joblib.dump(clf2, '../models/image_classifier.pkl')
print("\nModel saved!")

Improved Accuracy: 0.525

Classification Report:
              precision    recall  f1-score   support

       clean       0.50      0.54      0.52        96
       stego       0.55      0.51      0.53       104

    accuracy                           0.53       200
   macro avg       0.53      0.53      0.52       200
weighted avg       0.53      0.53      0.53       200


Model saved!


In [4]:
import os

clean_folder = '../data/images/train/train/clean'
stego_folder = '../data/images/train/train/stego'

clean_files = os.listdir(clean_folder)[:5]
stego_files = os.listdir(stego_folder)[:5]

print("Clean files:", clean_files)
print("Stego files:", stego_files)
print("\nTotal clean:", len(os.listdir(clean_folder)))
print("Total stego:", len(os.listdir(stego_folder)))

Clean files: ['00001.png', '00002.png', '00003.png', '00004.png', '00005.png']
Stego files: ['image_00001_eth_0.png', 'image_00001_eth_1.png', 'image_00001_ps_0.png', 'image_00002_eth_0.png', 'image_00002_html_0.png']

Total clean: 4000
Total stego: 12000


In [5]:
import cv2
import numpy as np
import pandas as pd
import os
import joblib
from scipy.fftpack import dct
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

def extract_features_v2(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (128, 128))
    dct_coeffs = dct(dct(img.T, norm='ortho').T, norm='ortho')
    return {
        'mean': np.mean(dct_coeffs),
        'std': np.std(dct_coeffs),
        'variance': np.var(dct_coeffs),
        'skewness': float(np.mean((dct_coeffs - np.mean(dct_coeffs))**3)),
        'energy': np.sum(dct_coeffs**2),
        'entropy': -np.sum((np.abs(dct_coeffs)/np.sum(np.abs(dct_coeffs)+1e-10)) *
                   np.log2(np.abs(dct_coeffs)/np.sum(np.abs(dct_coeffs)+1e-10) + 1e-10)),
        'kurtosis': float(np.mean((dct_coeffs - np.mean(dct_coeffs))**4)),
        'max_coeff': np.max(np.abs(dct_coeffs)),
        'pixel_mean': np.mean(img),
        'pixel_std': np.std(img)
    }

clean_folder = '../data/images/train/train/clean'
stego_folder = '../data/images/train/train/stego'

# Get matched pairs - use first 1000 clean images
clean_files = sorted(os.listdir(clean_folder))[:1000]
stego_files = sorted(os.listdir(stego_folder))

# Build a lookup of stego files by base number
stego_lookup = {}
for f in stego_files:
    # extract number e.g. '00001' from 'image_00001_eth_0.png'
    parts = f.split('_')
    if len(parts) >= 2:
        num = parts[1]
        if num not in stego_lookup:
            stego_lookup[num] = f

rows = []

# Add clean images
for f in clean_files:
    num = f.replace('.png', '')  # e.g. '00001'
    path = os.path.join(clean_folder, f)
    feats = extract_features_v2(path)
    if feats:
        feats['label'] = 0
        rows.append(feats)

# Add matched stego images
for f in clean_files:
    num = f.replace('.png', '')
    if num in stego_lookup:
        path = os.path.join(stego_folder, stego_lookup[num])
        feats = extract_features_v2(path)
        if feats:
            feats['label'] = 1
            rows.append(feats)

df3 = pd.DataFrame(rows)
print(f"Dataset: {len(df3)} images, {df3['label'].value_counts().to_dict()}")

X = df3.drop('label', axis=1)
y = df3['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf3 = RandomForestClassifier(n_estimators=200, random_state=42)
clf3.fit(X_train, y_train)

y_pred = clf3.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['clean', 'stego']))

joblib.dump(clf3, '../models/image_classifier.pkl')
print("\nModel saved!")

Dataset: 2000 images, {0: 1000, 1: 1000}
Accuracy: 0.0925

Classification Report:
              precision    recall  f1-score   support

       clean       0.08      0.08      0.08       199
       stego       0.11      0.11      0.11       201

    accuracy                           0.09       400
   macro avg       0.09      0.09      0.09       400
weighted avg       0.09      0.09      0.09       400


Model saved!


In [6]:
# Check what numbers we're actually extracting
stego_files = sorted(os.listdir('../data/images/train/train/stego'))[:5]
clean_files = sorted(os.listdir('../data/images/train/train/clean'))[:5]

print("Clean files:", clean_files)
print("Stego files:", stego_files)

# Check what the split produces
for f in stego_files:
    parts = f.split('_')
    print(f"File: {f} → parts: {parts} → extracted num: {parts[1] if len(parts)>=2 else 'FAIL'}")

Clean files: ['00001.png', '00002.png', '00003.png', '00004.png', '00005.png']
Stego files: ['image_00001_eth_0.png', 'image_00001_eth_1.png', 'image_00001_ps_0.png', 'image_00002_eth_0.png', 'image_00002_html_0.png']
File: image_00001_eth_0.png → parts: ['image', '00001', 'eth', '0.png'] → extracted num: 00001
File: image_00001_eth_1.png → parts: ['image', '00001', 'eth', '1.png'] → extracted num: 00001
File: image_00001_ps_0.png → parts: ['image', '00001', 'ps', '0.png'] → extracted num: 00001
File: image_00002_eth_0.png → parts: ['image', '00002', 'eth', '0.png'] → extracted num: 00002
File: image_00002_html_0.png → parts: ['image', '00002', 'html', '0.png'] → extracted num: 00002


In [7]:
import cv2
import numpy as np
import pandas as pd
import os
import joblib
from scipy.fftpack import dct
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

def extract_features_v2(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (128, 128))
    dct_coeffs = dct(dct(img.T, norm='ortho').T, norm='ortho')
    return {
        'mean': np.mean(dct_coeffs),
        'std': np.std(dct_coeffs),
        'variance': np.var(dct_coeffs),
        'skewness': float(np.mean((dct_coeffs - np.mean(dct_coeffs))**3)),
        'energy': np.sum(dct_coeffs**2),
        'entropy': -np.sum((np.abs(dct_coeffs)/np.sum(np.abs(dct_coeffs)+1e-10)) *
                   np.log2(np.abs(dct_coeffs)/np.sum(np.abs(dct_coeffs)+1e-10) + 1e-10)),
        'kurtosis': float(np.mean((dct_coeffs - np.mean(dct_coeffs))**4)),
        'max_coeff': np.max(np.abs(dct_coeffs)),
        'pixel_mean': np.mean(img),
        'pixel_std': np.std(img)
    }

clean_folder = '../data/images/train/train/clean'
stego_folder = '../data/images/train/train/stego'

rows = []

# Add ALL clean images with label 0
for f in sorted(os.listdir(clean_folder))[:1000]:
    feats = extract_features_v2(os.path.join(clean_folder, f))
    if feats:
        feats['label'] = 0
        rows.append(feats)

# Add ALL stego images with label 1
for f in sorted(os.listdir(stego_folder))[:1000]:
    feats = extract_features_v2(os.path.join(stego_folder, f))
    if feats:
        feats['label'] = 1
        rows.append(feats)

df = pd.DataFrame(rows)

# Verify labels are correct
print("Label distribution:", df['label'].value_counts().to_dict())
print("First 3 rows label:", df['label'].head(3).tolist())
print("Last 3 rows label:", df['label'].tail(3).tolist())

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['clean', 'stego']))

joblib.dump(clf, '../models/image_classifier.pkl')
print("\nModel saved!")

Label distribution: {0: 1000, 1: 1000}
First 3 rows label: [0, 0, 0]
Last 3 rows label: [1, 1, 1]

Accuracy: 0.57

Classification Report:
              precision    recall  f1-score   support

       clean       0.58      0.49      0.53       199
       stego       0.56      0.65      0.60       201

    accuracy                           0.57       400
   macro avg       0.57      0.57      0.57       400
weighted avg       0.57      0.57      0.57       400


Model saved!


In [8]:
import cv2
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

def extract_srm_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    if img is None:
        return None
    img = cv2.resize(img, (128, 128))

    # SRM high-pass filters — detect subtle pixel changes from embedding
    kernel1 = np.array([[0,0,0],[0,-1,1],[0,0,0]], dtype=np.float32)
    kernel2 = np.array([[0,0,0],[0,-1,0],[0,1,0]], dtype=np.float32)
    kernel3 = np.array([[0,0,0],[1,-2,1],[0,0,0]], dtype=np.float32)
    kernel4 = np.array([[0,1,0],[0,-2,0],[0,1,0]], dtype=np.float32)
    kernel5 = np.array([[1,0,-1],[0,0,0],[-1,0,1]], dtype=np.float32)

    feats = {}
    for i, k in enumerate([kernel1, kernel2, kernel3, kernel4, kernel5]):
        residual = cv2.filter2D(img, -1, k)
        feats[f'res{i}_mean'] = np.mean(residual)
        feats[f'res{i}_std'] = np.std(residual)
        feats[f'res{i}_energy'] = np.sum(residual**2)
        feats[f'res{i}_kurtosis'] = float(np.mean((residual - np.mean(residual))**4))
        feats[f'res{i}_skew'] = float(np.mean((residual - np.mean(residual))**3))

    # Also add pixel-level stats
    feats['pixel_mean'] = np.mean(img)
    feats['pixel_std'] = np.std(img)
    feats['pixel_var'] = np.var(img)

    return feats

clean_folder = '../data/images/train/train/clean'
stego_folder = '../data/images/train/train/stego'

rows = []

print("Loading clean images...")
for f in sorted(os.listdir(clean_folder))[:1500]:
    feats = extract_srm_features(os.path.join(clean_folder, f))
    if feats:
        feats['label'] = 0
        rows.append(feats)

print("Loading stego images...")
for f in sorted(os.listdir(stego_folder))[:1500]:
    feats = extract_srm_features(os.path.join(stego_folder, f))
    if feats:
        feats['label'] = 1
        rows.append(feats)

df = pd.DataFrame(rows)
print(f"Dataset: {len(df)} images, {df['label'].value_counts().to_dict()}")

X = df.drop('label', axis=1)
y = df['label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['clean', 'stego']))

joblib.dump(clf, '../models/image_classifier_srm.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print("\nModel and scaler saved!")

Loading clean images...
Loading stego images...
Dataset: 3000 images, {0: 1500, 1: 1500}

Accuracy: 0.695

Classification Report:
              precision    recall  f1-score   support

       clean       0.71      0.69      0.70       313
       stego       0.68      0.70      0.69       287

    accuracy                           0.69       600
   macro avg       0.69      0.70      0.69       600
weighted avg       0.70      0.69      0.70       600


Model and scaler saved!


In [1]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Try SVM on the same data
svm_clf = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_clf.fit(X_train, y_train)

y_pred_svm = svm_clf.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=['clean', 'stego']))

joblib.dump(svm_clf, '../models/image_classifier_svm.pkl')
print("\nSVM model saved!")

c:\Users\lizag\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\lizag\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


NameError: name 'X_train' is not defined

In [2]:
import cv2
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

def extract_srm_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    if img is None:
        return None
    img = cv2.resize(img, (128, 128))
    kernel1 = np.array([[0,0,0],[0,-1,1],[0,0,0]], dtype=np.float32)
    kernel2 = np.array([[0,0,0],[0,-1,0],[0,1,0]], dtype=np.float32)
    kernel3 = np.array([[0,0,0],[1,-2,1],[0,0,0]], dtype=np.float32)
    kernel4 = np.array([[0,1,0],[0,-2,0],[0,1,0]], dtype=np.float32)
    kernel5 = np.array([[1,0,-1],[0,0,0],[-1,0,1]], dtype=np.float32)
    feats = {}
    for i, k in enumerate([kernel1, kernel2, kernel3, kernel4, kernel5]):
        residual = cv2.filter2D(img, -1, k)
        feats[f'res{i}_mean'] = np.mean(residual)
        feats[f'res{i}_std'] = np.std(residual)
        feats[f'res{i}_energy'] = np.sum(residual**2)
        feats[f'res{i}_kurtosis'] = float(np.mean((residual - np.mean(residual))**4))
        feats[f'res{i}_skew'] = float(np.mean((residual - np.mean(residual))**3))
    feats['pixel_mean'] = np.mean(img)
    feats['pixel_std'] = np.std(img)
    feats['pixel_var'] = np.var(img)
    return feats

clean_folder = '../data/images/train/train/clean'
stego_folder = '../data/images/train/train/stego'

rows = []
print("Loading clean images...")
for f in sorted(os.listdir(clean_folder))[:1500]:
    feats = extract_srm_features(os.path.join(clean_folder, f))
    if feats:
        feats['label'] = 0
        rows.append(feats)

print("Loading stego images...")
for f in sorted(os.listdir(stego_folder))[:1500]:
    feats = extract_srm_features(os.path.join(stego_folder, f))
    if feats:
        feats['label'] = 1
        rows.append(feats)

df = pd.DataFrame(rows)
X = df.drop('label', axis=1)
y = df['label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Random Forest
print("\nTraining Random Forest...")
rf_clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

# SVM
print("\nTraining SVM...")
svm_clf = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_clf.fit(X_train, y_train)
svm_pred = svm_clf.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print("\nSVM Classification Report:")
print(classification_report(y_test, svm_pred, target_names=['clean', 'stego']))

# Save best model
joblib.dump(svm_clf, '../models/image_classifier_svm.pkl')
joblib.dump(rf_clf, '../models/image_classifier_srm.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print("\nAll models saved!")

Loading clean images...
Loading stego images...

Training Random Forest...
Random Forest Accuracy: 0.695

Training SVM...
SVM Accuracy: 0.71

SVM Classification Report:
              precision    recall  f1-score   support

       clean       0.89      0.50      0.64       313
       stego       0.63      0.93      0.75       287

    accuracy                           0.71       600
   macro avg       0.76      0.72      0.70       600
weighted avg       0.77      0.71      0.70       600


All models saved!
